# XAI stability -- TESTS

In [ ]:
import sys

!{sys.executable} -m pip install nbimporter
!{sys.executable} -m pip install tensorflow
!{sys.executable} -m pip install torch

#!git clone https://github.com/AI4LIFE-GROUP/OpenXAI.git
!{sys.executable} -m pip install -e OpenXAI

In [2]:
import time
import numpy as np
import pandas as pd
import nbimporter

# Utils
import torch
import os
import pickle
from sklearn.base import clone

import xgboost as xgb
from sklearn.neural_network import MLPClassifier


import Taylor_Explainer as texp
import XAI_stability_metrics as stab


import sklearn.ensemble
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [3]:
import openxai

# Data loaders
from openxai.dataloader import return_loaders

# Perturbation methods required for the computation of the relative stability metrics
from openxai.explainers.catalog.perturbation_methods import NormalPerturbation
from openxai.explainers.catalog.perturbation_methods import NewDiscrete_NormalPerturbation

In [4]:
import warnings
warnings.filterwarnings("ignore", message='should_run_async')

# Perturbation definition

In [5]:
# Perturbation class parameters
perturbation_mean= 0.0
perturbation_std= 0.05
perturbation_flip_percentage= 0.01
    
perturbation= NormalPerturbation('tabular',
                                 mean=perturbation_mean,
                                 std_dev=perturbation_std,
                                 flip_percentage=perturbation_flip_percentage)

def generate_mask(explanation, top_k):
    mask_indices= torch.topk(explanation, top_k).indices
    mask= torch.zeros(explanation.shape) > 10
    for i in mask_indices:
        mask[i]= True
    return mask

# Loading data

# 1. Synthetic - 20 features - Numeric

In [6]:
ox_path= 'data/synth_OX_20/processed/'

train_ox= pd.read_csv(ox_path + 'X_train.csv')
test_ox = pd.read_csv(ox_path + 'X_test.csv')
labels_train_ox= pd.read_csv(ox_path + 'y_train.csv')
labels_test_ox = pd.read_csv(ox_path + 'y_test.csv')

In [7]:
# 2024-05-29 23:24:10,338 Best: 0.908399 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_ox= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=256,
                            hidden_layer_sizes=(256,),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_ox.fit(train_ox, labels_train_ox.values.ravel())

acc_nn1_ox= sklearn.metrics.accuracy_score(labels_test_ox.values.ravel(), nn1_model_ox.predict(test_ox))
acc_nn1_ox

0.83

In [8]:
# 2024-05-31 17:29:14,888 Best: 0.912651 using {'batch_size': 64, 'lr': 0.01, 'max_epochs': 64, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_ox= MLPClassifier(batch_size= 64,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_ox.fit(train_ox, labels_train_ox.values.ravel())

acc_nn2_ox= sklearn.metrics.accuracy_score(labels_test_ox.values.ravel(), nn2_model_ox.predict(test_ox))
acc_nn2_ox

0.83

In [9]:
# 2024-06-03 16:27:56,409 Best: 0.905554 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_ox= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_ox.fit(train_ox, labels_train_ox.values.ravel())

acc_nn3_ox= sklearn.metrics.accuracy_score(labels_test_ox.values.ravel(), nn3_model_ox.predict(test_ox))
acc_nn3_ox

0.84

In [10]:
# definitions ---- 154 samples from test dataset

# get n and m parameters from test and labels_test
n_ox, m_ox= texp.get_n_m_sizes(test_ox.loc[0:153], labels_test_ox[0:153])

# conversion of test_ox data back to tensor as required by benchmarking methods
# explanations will be only done over test dataset
tn_test= torch.from_numpy((test_ox.loc[0:153]).values)


# define a descriptor to synthetic data
descriptor_ox= dict()

# h_min_dist_ox comes from the entire train data for capturing a model's space characteristic
h_min_dist_ox= texp.get_minimum_distance(train_ox)

# T-Exp explanation settings
descriptor_ox['h_min']= h_min_dist_ox
descriptor_ox['h_max']= 1
descriptor_ox['jacobian_eps']= 1e-3
descriptor_ox['max_itr']= 30
descriptor_ox['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_ox['num_samples']= 30
descriptor_ox['num_perts']= 10   # RIS/ROS
descriptor_ox['pert_max_distance']= (h_min_dist_ox/2)
descriptor_ox['num_runs']= 10    # RES
descriptor_ox['feature_metadata']= ['c'] * n_ox
descriptor_ox['p_norm']= 2
descriptor_ox['eps_norm']= 1e-6
descriptor_ox['top_k']= 0
descriptor_ox['mask']= generate_mask(tn_test[0].reshape(-1), descriptor_ox['top_k'])

In [11]:
h_min_dist_ox

0.2591968090373671

In [48]:
# ---- 154 samples from test dataset

# evaluate relative input/output stability -- using instances from train_ox
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_ox= stab.relative_stability(nn1_model_ox, test_ox[0:153], labels_test_ox[0:153], perturbation, 
                                        descriptor_ox, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn1_ox

RIS/ROS --
Progress: [████████████████████████████████████████] 153/153 - Est wait 00:0.05


--- 7269.77 seconds ---


{'t_exp_ris_max': 4821225.670931298,
 'std(t_exp_ris_max)': 442986.58323487476,
 'shap_ris_max': 204374.12695417175,
 'std(shap_ris_max)': 22323.661407022602,
 'lime_ris_max': 1237.5795620608358,
 'std(lime_ris_max)': 105.59632198111888,
 't_exp_ris_mean': 14293.907755382706,
 'std(t_exp_ris_mean)': 134769.7474725801,
 'shap_ris_mean': 1915.5949857766175,
 'std(shap_ris_mean)': 7145.294946654514,
 'lime_ris_mean': 8.702633068589284,
 'std(lime_ris_mean)': 43.12170871964754,
 't_exp_ros_max': 3116416835887.61,
 'std(t_exp_ros_max)': 251584665961.05005,
 'shap_ros_max': 67423332.58203492,
 'std(shap_ros_max)': 7544781.026784734,
 'lime_ros_max': 4035560.3427367015,
 'std(lime_ros_max)': 471855.61569745385,
 't_exp_ros_mean': 5385657508.928215,
 'std(t_exp_ros_mean)': 63263040734.83293,
 'shap_ros_mean': 529464.0743102329,
 'std(shap_ros_mean)': 2669350.3737353273,
 'lime_ros_mean': 38463.53408642695,
 'std(lime_ros_mean)': 192724.71390354098,
 'shap_kernel_ris_max': 515.8624388851616,
 '

In [50]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_ox
start= time.time()
print('RES --')
res_nn1_ox= stab.run_stability(nn1_model_ox, test_ox[0:153], labels_test_ox[0:153], 
                               descriptor_ox, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn1_ox

RES --
Progress: [████████████████████████████████████████] 153/153 - Est wait 00:0.08


--- 6554.0 seconds ---


{'texp_res': 9.455164053283262e-15,
 'shap_res': 0.1654364387847423,
 'shap_kernel_res': 0.050805465402052205,
 'shap_exact_res': '--',
 'lime_res': 5.801606246868625e-17,
 'itGd_res': 1.1149867635186251e-14,
 'iXGd_res': 8.145812e-06,
 'dLif_res': 8.302823e-06,
 'lwrp_res': 8.777078e-06,
 'smoothG_res': 7.418584719782159,
 'vanillaG_res': 1.3662861e-05,
 'GuidBprop_res': 1.3662861e-05,
 'occlusion_res': 5.2452087e-06}

In [52]:
# evaluate relative input/output stability -- using instances from train_ox
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_ox= stab.relative_stability(nn2_model_ox, test_ox[0:153], labels_test_ox[0:153], perturbation, 
                                        descriptor_ox, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn2_ox

RIS/ROS --
Progress: [████████████████████████████████████████] 153/153 - Est wait 00:0.08


--- 13891.01 seconds ---


{'t_exp_ris_max': 7445.926911892156,
 'std(t_exp_ris_max)': 676.1006317503426,
 'shap_ris_max': 183480.97527580627,
 'std(shap_ris_max)': 21024.364581474805,
 'lime_ris_max': 246.9955027229364,
 'std(lime_ris_max)': 29.235202739895822,
 't_exp_ris_mean': 66.395604943081,
 'std(t_exp_ris_mean)': 388.4526638240421,
 'shap_ris_mean': 2014.5467897581593,
 'std(shap_ris_mean)': 7014.007642742592,
 'lime_ris_mean': 4.1459024735821695,
 'std(lime_ris_mean)': 12.949136715149248,
 't_exp_ros_max': 13202.027275677428,
 'std(t_exp_ros_max)': 1870.4753833187517,
 'shap_ros_max': 21735784.295696005,
 'std(shap_ros_max)': 1758030.1215365182,
 'lime_ros_max': 2273.132319237589,
 'std(lime_ros_max)': 227.2221686079457,
 't_exp_ros_mean': 145.81948976646805,
 'std(t_exp_ros_mean)': 557.4193655455662,
 'shap_ros_mean': 18951.671515514776,
 'std(shap_ros_mean)': 184743.9606707885,
 'lime_ros_mean': 10.574411506111215,
 'std(lime_ros_mean)': 30.383799887753405,
 'shap_kernel_ris_max': 2610.1153208000533,


In [54]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_ox
start= time.time()
print('RES --')
res_nn2_ox= stab.run_stability(nn2_model_ox, test_ox[0:153], labels_test_ox[0:153], 
                               descriptor_ox, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn2_ox

RES --
Progress: [████████████████████████████████████████] 153/153 - Est wait 00:0.07


--- 13569.73 seconds ---


{'texp_res': 4.0943002132167226e-15,
 'shap_res': 0.16666805207757526,
 'shap_kernel_res': 0.05145518255482718,
 'shap_exact_res': '--',
 'lime_res': 5.3617058706823765e-17,
 'itGd_res': 7.838739178637249e-15,
 'iXGd_res': 3.1411776e-06,
 'dLif_res': 3.1047741e-06,
 'lwrp_res': 2.6744574e-06,
 'smoothG_res': 4.707600141205531,
 'vanillaG_res': 7.0414253e-06,
 'GuidBprop_res': 7.0414253e-06,
 'occlusion_res': 2.311554e-06}

In [56]:
# evaluate relative input/output stability -- using instances from train_ox
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_ox= stab.relative_stability(nn3_model_ox, test_ox[0:153], labels_test_ox[0:153], perturbation, 
                                        descriptor_ox, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn3_ox

RIS/ROS --
Progress: [████████████████████████████████████████] 153/153 - Est wait 00:0.04


--- 17831.67 seconds ---


{'t_exp_ris_max': 6461.602453857154,
 'std(t_exp_ris_max)': 535.1630888741596,
 'shap_ris_max': 178944.91182446218,
 'std(shap_ris_max)': 23758.19396955374,
 'lime_ris_max': 393.00344807626084,
 'std(lime_ris_max)': 31.7162306571867,
 't_exp_ris_mean': 42.37709694749165,
 'std(t_exp_ris_mean)': 204.090024107523,
 'shap_ris_mean': 2954.757658954058,
 'std(shap_ris_mean)': 10958.505366359746,
 'lime_ris_mean': 1.9160579424721909,
 'std(lime_ris_mean)': 11.283053386014524,
 't_exp_ros_max': 525917.6196032815,
 'std(t_exp_ros_max)': 42377.63459648172,
 'shap_ros_max': 2882705.3256835258,
 'std(shap_ros_max)': 288546.5901637205,
 'lime_ros_max': 3724.3079568898393,
 'std(lime_ros_max)': 321.1064024221361,
 't_exp_ros_mean': 492.9509768037587,
 'std(t_exp_ros_mean)': 4952.36751199208,
 'shap_ros_mean': 8133.24091363203,
 'std(shap_ros_mean)': 37004.70402471637,
 'lime_ros_mean': 8.980310771216674,
 'std(lime_ros_mean)': 41.77409341479055,
 'shap_kernel_ris_max': 1035.9000103337507,
 'std(sha

In [58]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_ox
start= time.time()
print('RES --')
res_nn3_ox= stab.run_stability(nn3_model_ox, test_ox[0:153], labels_test_ox[0:153], 
                               descriptor_ox, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn3_ox

RES --
Progress: [████████████████████████████████████████] 153/153 - Est wait 00:0.028


--- 18513.99 seconds ---


{'texp_res': 5.630757419431282e-15,
 'shap_res': 0.1594958964118583,
 'shap_kernel_res': 0.04835806581704387,
 'shap_exact_res': '--',
 'lime_res': 5.793820594181519e-17,
 'itGd_res': 5.208590938883218e-15,
 'iXGd_res': 4.5775437e-06,
 'dLif_res': 4.874844e-06,
 'lwrp_res': 4.572496e-06,
 'smoothG_res': 6.439052111461263,
 'vanillaG_res': 7.6779015e-06,
 'GuidBprop_res': 7.6779015e-06,
 'occlusion_res': 2.5788913e-06}

# TODO LIST
# - Setar parâmetros dos modelos de 2., 3., 8., 10.
# - Verificar h_min de todos os conjuntos antes de testar e fixar um h_mim, caso necessário

# DONE -- synth, diabetes, german, compas, independent
# OPEN -- heloc

# 2. Adult Income

In [ ]:
ad_path= 'data/adult/processed/'

train_ad= pd.read_csv(ad_path + 'X_train.csv')
test_ad = pd.read_csv(ad_path + 'X_test.csv')
labels_train_ad= pd.read_csv(ad_path + 'y_train.csv')
labels_test_ad = pd.read_csv(ad_path + 'y_test.csv')

In [ ]:
# 2024-05-29 23:24:10,338 Best: 0.908399 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_ad= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=256,
                            hidden_layer_sizes=(256,),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_ad.fit(train_ad, labels_train_ad.values.ravel())

acc_nn1_ad= sklearn.metrics.accuracy_score(labels_test_ad.values.ravel(), nn1_model_ad.predict(test_ad))
acc_nn1_ad

In [ ]:
# 2024-05-31 17:29:14,888 Best: 0.912651 using {'batch_size': 64, 'lr': 0.01, 'max_epochs': 64, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_ad= MLPClassifier(batch_size= 64,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_ad.fit(train_ad, labels_train_ad.values.ravel())

acc_nn2_ad= sklearn.metrics.accuracy_score(labels_test_ad.values.ravel(), nn2_model_ad.predict(test_ad))
acc_nn2_ad

In [ ]:
# 2024-06-03 16:27:56,409 Best: 0.905554 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_ad= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_ad.fit(train_ad, labels_train_ad.values.ravel())

acc_nn3_ad= sklearn.metrics.accuracy_score(labels_test_ad.values.ravel(), nn3_model_ad.predict(test_ad))
acc_nn3_ad

In [ ]:
# definitions ---- 624 samples from test dataset

# get n and m parameters from test and labels_test
n_ad, m_ad= texp.get_n_m_sizes(test_ad.loc[0:623], labels_test_ad[0:623])

# conversion of test_ox data back to tensor as required by benchmarking methods
# explanations will be only done over test dataset
tn_test= torch.from_numpy((test_ad.loc[0:623]).values)


# define a descriptor to synthetic data
descriptor_ad= dict()

# h_min_dist_ox comes from the entire train data for capturing a model's space characteristic
h_min_dist_ad= texp.get_minimum_distance(train_ad)

# T-Exp explanation settings
descriptor_ad['h_min']= h_min_dist_ad
descriptor_ad['h_max']= 1
descriptor_ad['jacobian_eps']= 1e-3
descriptor_ad['max_itr']= 30
descriptor_ad['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_ad['num_samples']= 30
descriptor_ad['num_perts']= 10   # RIS/ROS
descriptor_ad['pert_max_distance']= (h_min_dist_ad/2)
descriptor_ad['num_runs']= 10    # RES
descriptor_ad['feature_metadata']= ['c'] * n_ad
descriptor_ad['p_norm']= 2
descriptor_ad['eps_norm']= 1e-6
descriptor_ad['top_k']= 0
descriptor_ad['mask']= generate_mask(tn_test[0].reshape(-1), descriptor_ad['top_k'])

In [ ]:
# ---- 624 samples from test dataset

# evaluate relative input/output stability -- using instances from train_ad
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_ad= stab.relative_stability(nn1_model_ad, test_ad[0:623], labels_test_ad[0:623], perturbation, 
                                        descriptor_ad, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn1_ad

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_ad
start= time.time()
print('RES --')
res_nn1_ad= stab.run_stability(nn1_model_ad, test_ad[0:623], labels_test_ad[0:623], 
                               descriptor_ad, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn1_ad

In [ ]:
# evaluate relative input/output stability -- using instances from train_ad
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_ad= stab.relative_stability(nn2_model_ad, test_ad[0:623], labels_test_ad[0:623], perturbation, 
                                        descriptor_ad, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn2_ad

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_ad
start= time.time()
print('RES --')
res_nn2_ad= stab.run_stability(nn2_model_ad, test_ad[0:623], labels_test_ad[0:623], 
                               descriptor_ad, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn2_ad

In [ ]:
# evaluate relative input/output stability -- using instances from train_ad
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_ad= stab.relative_stability(nn3_model_ad, test_ad[0:623], labels_test_ad[0:623], perturbation, 
                                        descriptor_ad, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn3_ad

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_ad
start= time.time()
print('RES --')
res_nn3_ad= stab.run_stability(nn3_model_ad, test_ad[0:623], labels_test_ad[0:623], 
                               descriptor_ad, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn3_ad

# 3. Chess (kr-vs-kp)

In [ ]:
ch_path= 'data/chess/processed/'

train_ch= pd.read_csv(ch_path + 'X_train.csv')
test_ch = pd.read_csv(ch_path + 'X_test.csv')
labels_train_ch= pd.read_csv(ch_path + 'y_train.csv')
labels_test_ch = pd.read_csv(ch_path + 'y_test.csv')

In [ ]:
# 2024-05-29 23:24:10,338 Best: 0.908399 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_ch= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=256,
                            hidden_layer_sizes=(256,),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_ch.fit(train_ch, labels_train_ch.values.ravel())

acc_nn1_ch= sklearn.metrics.accuracy_score(labels_test_ch.values.ravel(), nn1_model_ch.predict(test_ch))
acc_nn1_ch

In [ ]:
# 2024-05-31 17:29:14,888 Best: 0.912651 using {'batch_size': 64, 'lr': 0.01, 'max_epochs': 64, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_ch= MLPClassifier(batch_size= 64,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_ch.fit(train_ch, labels_train_ch.values.ravel())

acc_nn2_ch= sklearn.metrics.accuracy_score(labels_test_ch.values.ravel(), nn2_model_ch.predict(test_ch))
acc_nn2_ch

In [ ]:
# 2024-06-03 16:27:56,409 Best: 0.905554 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_ch= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_ch.fit(train_ch, labels_train_ch.values.ravel())

acc_nn3_ch= sklearn.metrics.accuracy_score(labels_test_ch.values.ravel(), nn3_model_ch.predict(test_ch))
acc_nn3_ch

In [ ]:
# definitions ---- 327 samples from test dataset

# get n and m parameters from test and labels_test
n_ch, m_ch= texp.get_n_m_sizes(test_ch.loc[0:326], labels_test_ch[0:326])

# conversion of test_ox data back to tensor as required by benchmarking methods
# explanations will be only done over test dataset
tn_test= torch.from_numpy((test_ch.loc[0:326]).values)


# define a descriptor to synthetic data
descriptor_ch= dict()

# h_min_dist_ox comes from the entire train data for capturing a model's space characteristic
h_min_dist_ch= texp.get_minimum_distance(train_ch)

# T-Exp explanation settings
descriptor_ch['h_min']= h_min_dist_ch
descriptor_ch['h_max']= 1
descriptor_ch['jacobian_eps']= 1e-3
descriptor_ch['max_itr']= 30
descriptor_ch['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_ch['num_samples']= 30
descriptor_ch['num_perts']= 10   # RIS/ROS
descriptor_ch['pert_max_distance']= (h_min_dist_ch/2)
descriptor_ch['num_runs']= 10    # RES
descriptor_ch['feature_metadata']= ['c'] * n_ch
descriptor_ch['p_norm']= 2
descriptor_ch['eps_norm']= 1e-6
descriptor_ch['top_k']= 0
descriptor_ch['mask']= generate_mask(tn_test[0].reshape(-1), descriptor_ch['top_k'])

In [ ]:
# ---- 327 samples from test dataset

# evaluate relative input/output stability -- using instances from train_ch
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_ch= stab.relative_stability(nn1_model_ch, test_ch[0:326], labels_test_ch[0:326], perturbation, 
                                        descriptor_ch, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn1_ch

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_ch
start= time.time()
print('RES --')
res_nn1_ch= stab.run_stability(nn1_model_ch, test_ch[0:326], labels_test_ch[0:326], 
                               descriptor_ch, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn1_ch

In [ ]:
# evaluate relative input/output stability -- using instances from train_ch
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_ch= stab.relative_stability(nn2_model_ch, test_ch[0:326], labels_test_ch[0:326], perturbation, 
                                        descriptor_ch, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn2_ch

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_ch
start= time.time()
print('RES --')
res_nn2_ch= stab.run_stability(nn2_model_ch, test_ch[0:326], labels_test_ch[0:326], 
                               descriptor_ch, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn2_ch

In [ ]:
# evaluate relative input/output stability -- using instances from train_ch
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_ch= stab.relative_stability(nn3_model_ch, test_ch[0:326], labels_test_ch[0:326], perturbation, 
                                        descriptor_ch, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn3_ch

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_ch
start= time.time()
print('RES --')
res_nn3_ch= stab.run_stability(nn3_model_ch, test_ch[0:326], labels_test_ch[0:326], 
                               descriptor_ch, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn3_ch

# 4. COMPAS

In [14]:
cpas_path= 'data/compas/processed/'

train_cpas= pd.read_csv(cpas_path + 'X_train.csv')
test_cpas = pd.read_csv(cpas_path + 'X_test.csv')
labels_train_cpas= pd.read_csv(cpas_path + 'y_train.csv')
labels_test_cpas = pd.read_csv(cpas_path + 'y_test.csv')

In [15]:
numeric_columns_cpas= ['age', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count']

categor_columns_cpas= list(filter(lambda x:x not in numeric_columns_cpas, train_cpas.columns))

categor_columns_cpas

['sex',
 'age_cat_25-45',
 'age_cat_Greaterthan45',
 'age_cat_Lessthan25',
 'race_African-American',
 'race_Caucasian',
 'c_charge_degree_F',
 'c_charge_degree_M']

In [16]:
# 2024-05-30 04:34:50,307 Best: 0.729593 using {'batch_size': 32, 'lr': 0.01, 'max_epochs': 128, 
#                               'module__n_features': 13, 'module__n_neurons': 16, 'module__nonlin': Tanh()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_cpas= MLPClassifier(batch_size= 32,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(16,),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_cpas.fit(train_cpas, labels_train_cpas.values.ravel())

acc_nn1_cpas= sklearn.metrics.accuracy_score(labels_test_cpas.values.ravel(), nn1_model_cpas.predict(test_cpas))
acc_nn1_cpas

0.6903409090909091

In [17]:
# 2024-05-31 23:28:32,931 Best: 0.728194 using {'batch_size': 32, 'lr': 0.02, 'max_epochs': 32, 
#                               'module__n_features': 13, 'module__n_neurons': 32, 'module__nonlin': Tanh()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_cpas= MLPClassifier(batch_size= 32,
                            learning_rate='constant', learning_rate_init= 0.02,
                            max_iter=32,
                            hidden_layer_sizes=(32, 32),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_cpas.fit(train_cpas, labels_train_cpas.values.ravel())

acc_nn2_cpas= sklearn.metrics.accuracy_score(labels_test_cpas.values.ravel(), nn2_model_cpas.predict(test_cpas))
acc_nn2_cpas

0.6912878787878788

In [18]:
# 2024-06-03 22:58:26,596 Best: 0.728264 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 32, 
#                               'module__n_features': 13, 'module__n_neurons': 16, 'module__nonlin': Tanh()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_cpas= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=64,
                            hidden_layer_sizes=(16, 16, 16),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_cpas.fit(train_cpas, labels_train_cpas.values.ravel())

acc_nn3_cpas= sklearn.metrics.accuracy_score(labels_test_cpas.values.ravel(), nn3_model_cpas.predict(test_cpas))
acc_nn3_cpas

0.6856060606060606

In [19]:
# definitions ---- 409 samples from test dataset

# get n and m parameters from test and labels_test
n_cpas, m_cpas= texp.get_n_m_sizes(test_cpas.loc[0:408], labels_test_cpas[0:408])

# conversion of test_ox data back to tensor as required by benchmarking methods
# explanations will be only done over test dataset
tn_test= torch.from_numpy((test_cpas.loc[0:408]).values)


# define a descriptor to synthetic data
descriptor_cpas= dict()

# h_min_dist_ox comes from the entire train data for capturing a model's space characteristic
h_min_dist_cpas= texp.get_minimum_distance(train_cpas)

# T-Exp explanation settings
descriptor_cpas['h_min']= h_min_dist_cpas
descriptor_cpas['h_max']= 1
descriptor_cpas['jacobian_eps']= 1e-3
descriptor_cpas['max_itr']= 30
descriptor_cpas['ohe_delta']= 0.2

# perturbation settings used for the stability metric
descriptor_cpas['num_samples']= 30
descriptor_cpas['num_perts']= 10   # RIS/ROS
descriptor_cpas['pert_max_distance']= (h_min_dist_cpas/2)
descriptor_cpas['num_runs']= 10    # RES
descriptor_cpas['feature_metadata']= ['c'] * n_cpas
descriptor_cpas['p_norm']= 2
descriptor_cpas['eps_norm']= 1e-6
descriptor_cpas['top_k']= 0
descriptor_cpas['mask']= generate_mask(tn_test[0].reshape(-1), descriptor_cpas['top_k'])

In [20]:
if (h_min_dist_cpas< 0.01): h_min_dist_cpas= 0.01

h_min_dist_cpas

0.01612899999999995

In [22]:
# ---- 409 samples from test dataset

# evaluate relative input/output stability -- using instances from train_cpas
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_cpas= stab.relative_stability(nn1_model_cpas, test_cpas[0:408], labels_test_cpas[0:408], perturbation, 
                                        descriptor_cpas, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn1_cpas

RIS/ROS --
Progress: [████████████████████████████████████████] 408/408 - Est wait 00:0.003


--- 14285.83 seconds ---


{'t_exp_ris_max': 1488.036452967034,
 'std(t_exp_ris_max)': 112.28940479612095,
 'shap_ris_max': 147022.68792479992,
 'std(shap_ris_max)': 9645.113546796141,
 'lime_ris_max': 862.8896879451089,
 'std(lime_ris_max)': 59.07883144520098,
 't_exp_ris_mean': 12.236214140898742,
 'std(t_exp_ris_mean)': 32.61490445919983,
 'shap_ris_mean': 412.9697204663537,
 'std(shap_ris_mean)': 3383.96912852278,
 'lime_ris_mean': 1.769526718909519,
 'std(lime_ris_mean)': 13.35588086787295,
 't_exp_ros_max': 69190.00554728795,
 'std(t_exp_ros_max)': 3740.0169547560704,
 'shap_ros_max': 3167041.7328436133,
 'std(shap_ros_max)': 168638.66797756377,
 'lime_ros_max': 4775.416243665063,
 'std(lime_ros_max)': 391.30103548567155,
 't_exp_ros_mean': 74.71390962169265,
 'std(t_exp_ros_mean)': 380.27803397839574,
 'shap_ros_mean': 2243.335743731569,
 'std(shap_ros_mean)': 19178.323882739216,
 'lime_ros_mean': 10.238536166173658,
 'std(lime_ros_mean)': 52.82818860938806,
 'shap_kernel_ris_max': 257114.87477690692,
 's

In [24]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_cpas
start= time.time()
print('RES --')
res_nn1_cpas= stab.run_stability(nn1_model_cpas, test_cpas[0:408], labels_test_cpas[0:408], 
                               descriptor_cpas, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn1_cpas

RES --
Progress: [████████████████████████████████████████] 408/408 - Est wait 00:0.081


--- 11758.35 seconds ---


{'texp_res': 4.449557262054371e-16,
 'shap_res': 0.14748700507728352,
 'shap_kernel_res': 0.01602031793101775,
 'shap_exact_res': 1.1585084522564614e-16,
 'lime_res': 6.267851434100963e-17,
 'itGd_res': 9.028031637431707e-16,
 'iXGd_res': 5.4056517e-07,
 'dLif_res': 5.4384134e-07,
 'lwrp_res': 5.356133e-07,
 'smoothG_res': 0.638236263301794,
 'vanillaG_res': 1.9718022e-06,
 'GuidBprop_res': 1.9718022e-06,
 'occlusion_res': 5.380949e-07}

In [26]:
# evaluate relative input/output stability -- using instances from train_cpas
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_cpas= stab.relative_stability(nn2_model_cpas, test_cpas[0:408], labels_test_cpas[0:408], perturbation, 
                                        descriptor_cpas, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn2_cpas

RIS/ROS --
Progress: [████████████████████████████████████████] 408/408 - Est wait 00:0.031


--- 21317.86 seconds ---


{'t_exp_ris_max': 4050.133410534023,
 'std(t_exp_ris_max)': 346.18957358076005,
 'shap_ris_max': 102644.76171360198,
 'std(shap_ris_max)': 8253.783253944215,
 'lime_ris_max': 1261.899106679175,
 'std(lime_ris_max)': 63.59442473986166,
 't_exp_ris_mean': 36.65472391280354,
 'std(t_exp_ris_mean)': 106.90465801857376,
 'shap_ris_mean': 270.1680189129065,
 'std(shap_ris_mean)': 2021.0795984331066,
 'lime_ris_mean': 1.7643914705004697,
 'std(lime_ris_mean)': 10.366573324987455,
 't_exp_ros_max': 82782.89617549146,
 'std(t_exp_ros_max)': 7981.016365049732,
 'shap_ros_max': 1503893.5656908876,
 'std(shap_ros_max)': 76887.86748127553,
 'lime_ros_max': 13797.519958005712,
 'std(lime_ros_max)': 743.759693609947,
 't_exp_ros_mean': 458.02771709965117,
 'std(t_exp_ros_mean)': 2431.171990841877,
 'shap_ros_mean': 2020.8658543223003,
 'std(shap_ros_mean)': 13801.180037969581,
 'lime_ros_mean': 21.92532280970877,
 'std(lime_ros_mean)': 98.24557494406642,
 'shap_kernel_ris_max': 78536.14363161911,
 's

In [28]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_cpas
start= time.time()
print('RES --')
res_nn2_cpas= stab.run_stability(nn2_model_cpas, test_cpas[0:408], labels_test_cpas[0:408], 
                               descriptor_cpas, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn2_cpas

RES --
Progress: [████████████████████████████████████████] 408/408 - Est wait 00:0.059


--- 16518.07 seconds ---


{'texp_res': 1.5959455978986625e-16,
 'shap_res': 0.16921852881356464,
 'shap_kernel_res': 0.020363057549913865,
 'shap_exact_res': 1.3611362695753883e-16,
 'lime_res': 4.036727992921851e-17,
 'itGd_res': 3.589065315655259e-15,
 'iXGd_res': 1.9717029e-06,
 'dLif_res': 1.9716888e-06,
 'lwrp_res': 9.991837e-07,
 'smoothG_res': 1.27148086865479,
 'vanillaG_res': 4.0065006e-06,
 'GuidBprop_res': 4.0065006e-06,
 'occlusion_res': 1.0662403e-06}

In [30]:
# evaluate relative input/output stability -- using instances from train_cpas
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_cpas= stab.relative_stability(nn3_model_cpas, test_cpas[0:408], labels_test_cpas[0:408], perturbation, 
                                        descriptor_cpas, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn3_cpas

RIS/ROS --
Progress: [████████████████████████████████████████] 408/408 - Est wait 00:0.066


--- 23486.63 seconds ---


{'t_exp_ris_max': 1328010.9326359155,
 'std(t_exp_ris_max)': 65682.29107449598,
 'shap_ris_max': 82975.09812485453,
 'std(shap_ris_max)': 6272.927504806458,
 'lime_ris_max': 4940.301901903717,
 'std(lime_ris_max)': 285.40429081136267,
 't_exp_ris_mean': 452.7459477830925,
 'std(t_exp_ris_mean)': 7746.224376673039,
 'shap_ris_mean': 375.2244948947214,
 'std(shap_ris_mean)': 2148.41922023554,
 'lime_ris_mean': 17.182825999033458,
 'std(lime_ris_mean)': 94.37278934055585,
 't_exp_ros_max': 8218708.600076098,
 'std(t_exp_ros_max)': 410432.99141640594,
 'shap_ros_max': 32530341.663680725,
 'std(shap_ros_max)': 2348768.6692175977,
 'lime_ros_max': 1657429.9668715035,
 'std(lime_ros_max)': 96433.98062667003,
 't_exp_ros_mean': 4333.959407249064,
 'std(t_exp_ros_mean)': 61954.25933790909,
 'shap_ros_mean': 39156.183061747426,
 'std(shap_ros_mean)': 346821.31046903465,
 'lime_ros_mean': 1503.228177913701,
 'std(lime_ros_mean)': 11441.977873513184,
 'shap_kernel_ris_max': 41655.84663955457,
 'st

In [32]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_cpas
start= time.time()
print('RES --')
res_nn3_cpas= stab.run_stability(nn3_model_cpas, test_cpas[0:408], labels_test_cpas[0:408], 
                               descriptor_cpas, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn3_cpas

RES --
Progress: [████████████████████████████████████████] 408/408 - Est wait 00:0.054


--- 14892.27 seconds ---


{'texp_res': 2.3055512673781017e-16,
 'shap_res': 0.15399107618043936,
 'shap_kernel_res': 0.020872290038648975,
 'shap_exact_res': 1.3022581525859256e-16,
 'lime_res': 3.975238760931129e-17,
 'itGd_res': 3.5734697288860764e-15,
 'iXGd_res': 1.974165e-06,
 'dLif_res': 1.0991564e-06,
 'lwrp_res': 1.93325e-06,
 'smoothG_res': 1.7610481637801425,
 'vanillaG_res': 4.4246162e-06,
 'GuidBprop_res': 4.4246162e-06,
 'occlusion_res': 1.0323828e-06}

# 5. Diabetes

In [12]:
diab_path= 'data/diabetes/processed/'

train_diab= pd.read_csv(diab_path + 'X_train.csv')
test_diab = pd.read_csv(diab_path + 'X_test.csv')
labels_train_diab= pd.read_csv(diab_path + 'y_train.csv')
labels_test_diab = pd.read_csv(diab_path + 'y_test.csv')

In [13]:
# 2024-05-29 17:25:01,468 Best: 0.849094 using {'batch_size': 32, 'lr': 0.02, 'max_epochs': 128, 
#                               'module__n_features': 8, 'module__n_neurons': 64, 'module__nonlin': Tanh()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_diab= MLPClassifier(batch_size= 32,
                            learning_rate='constant', learning_rate_init= 0.02,
                            max_iter=128,
                            hidden_layer_sizes=(64,),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_diab.fit(train_diab, labels_train_diab.values.ravel())

acc_nn1_diab= sklearn.metrics.accuracy_score(labels_test_diab.values.ravel(), nn1_model_diab.predict(test_diab))
acc_nn1_diab

0.7532467532467533

In [14]:
# 2024-05-31 17:04:14,821 Best: 0.849508 using {'batch_size': 32, 'lr': 0.001, 'max_epochs': 128, 
#                               'module__n_features': 8, 'module__n_neurons': 128, 'module__nonlin': ReLU()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_diab= MLPClassifier(batch_size= 32,
                            learning_rate='constant', learning_rate_init= 0.001,
                            max_iter=512,
                            hidden_layer_sizes=(128, 128),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_diab.fit(train_diab, labels_train_diab.values.ravel())

acc_nn2_diab= sklearn.metrics.accuracy_score(labels_test_diab.values.ravel(), nn2_model_diab.predict(test_diab))
acc_nn2_diab

0.7142857142857143

In [15]:
# 2024-06-03 16:05:00,462 Best: 0.852499 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 16, 
#                               'module__n_features': 8, 'module__n_neurons': 64, 'module__nonlin': ReLU()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_diab= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(64, 64, 64),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_diab.fit(train_diab, labels_train_diab.values.ravel())

acc_nn3_diab= sklearn.metrics.accuracy_score(labels_test_diab.values.ravel(), nn3_model_diab.predict(test_diab))
acc_nn3_diab

0.7597402597402597

In [16]:
# definitions ---- 126 samples from test dataset

# get n and m parameters from test and labels_test
n_diab, m_diab= texp.get_n_m_sizes(test_diab.loc[0:125], labels_test_diab[0:125])

# conversion of test_ox data back to tensor as required by benchmarking methods
# explanations will be only done over test dataset
tn_test= torch.from_numpy((test_diab.loc[0:125]).values)


# define a descriptor to synthetic data
descriptor_diab= dict()

# h_min_dist_ox comes from the entire train data for capturing a model's space characteristic
h_min_dist_diab= texp.get_minimum_distance(train_diab)

# T-Exp explanation settings
descriptor_diab['h_min']= h_min_dist_diab
descriptor_diab['h_max']= 1
descriptor_diab['jacobian_eps']= 1e-3
descriptor_diab['max_itr']= 30
descriptor_diab['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_diab['num_samples']= 30
descriptor_diab['num_perts']= 10   # RIS/ROS
descriptor_diab['pert_max_distance']= (h_min_dist_diab/2)
descriptor_diab['num_runs']= 10    # RES
descriptor_diab['feature_metadata']= ['c'] * n_diab
descriptor_diab['p_norm']= 2
descriptor_diab['eps_norm']= 1e-6
descriptor_diab['top_k']= 0
descriptor_diab['mask']= generate_mask(tn_test[0].reshape(-1), descriptor_diab['top_k'])

In [17]:
h_min_dist_diab

0.06144666202958625

In [36]:
# ---- 126 samples from test dataset

# evaluate relative input/output stability -- using instances from train_diab
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_diab= stab.relative_stability(nn1_model_diab, test_diab[0:125], labels_test_diab[0:125], perturbation, 
                                        descriptor_diab, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn1_diab

RIS/ROS --
Progress: [████████████████████████████████████████] 125/125 - Est wait 00:0.04


--- 2175.48 seconds ---


{'t_exp_ris_max': 176.19925376161817,
 'std(t_exp_ris_max)': 17.33607284726268,
 'shap_ris_max': 275.7012254743256,
 'std(shap_ris_max)': 56.44894092295107,
 'lime_ris_max': 43.22111654430163,
 'std(lime_ris_max)': 4.223318293559915,
 't_exp_ris_mean': 3.041941039989577,
 'std(t_exp_ris_mean)': 7.346369911769342,
 'shap_ris_mean': 12.120899224522793,
 'std(shap_ris_mean)': 20.544279539496166,
 'lime_ris_mean': 0.7620162579914606,
 'std(lime_ris_mean)': 1.5171936786717553,
 't_exp_ros_max': 4802.208606045823,
 'std(t_exp_ros_max)': 526.8360490381579,
 'shap_ros_max': 41756.16686822898,
 'std(shap_ros_max)': 4006.362658743071,
 'lime_ros_max': 576.0830924225438,
 'std(lime_ros_max)': 83.75701133833402,
 't_exp_ros_mean': 27.48245591955461,
 'std(t_exp_ros_mean)': 62.25237192436754,
 'shap_ros_mean': 139.2815935936298,
 'std(shap_ros_mean)': 508.59473265722767,
 'lime_ros_mean': 5.614124152433269,
 'std(lime_ros_mean)': 10.671219707113881,
 'shap_kernel_ris_max': 275.701225474351,
 'std(s

In [38]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_diab
start= time.time()
print('RES --')
res_nn1_diab= stab.run_stability(nn1_model_diab, test_diab[0:125], labels_test_diab[0:125], 
                               descriptor_diab, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn1_diab

RES --
Progress: [████████████████████████████████████████] 125/125 - Est wait 00:0.04


--- 1926.72 seconds ---


{'texp_res': 9.490063172835475e-16,
 'shap_res': 2.2226980149236474e-16,
 'shap_kernel_res': 1.2566871346510768e-16,
 'shap_exact_res': 2.2226980149236474e-16,
 'lime_res': 6.397344083151129e-17,
 'itGd_res': 1.0310738718936457e-15,
 'iXGd_res': 5.4730396e-07,
 'dLif_res': 4.9623327e-07,
 'lwrp_res': 5.3312016e-07,
 'smoothG_res': 0.3173694714200193,
 'vanillaG_res': 1.2160663e-06,
 'GuidBprop_res': 1.2160663e-06,
 'occlusion_res': 5.462856e-07}

In [40]:
# evaluate relative input/output stability -- using instances from train_diab
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_diab= stab.relative_stability(nn2_model_diab, test_diab[0:125], labels_test_diab[0:125], perturbation, 
                                        descriptor_diab, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn2_diab

RIS/ROS --
Progress: [████████████████████████████████████████] 125/125 - Est wait 00:0.09


--- 3360.29 seconds ---


{'t_exp_ris_max': 906.0301945753729,
 'std(t_exp_ris_max)': 105.5852359370849,
 'shap_ris_max': 14158.358832070497,
 'std(shap_ris_max)': 1260.3601101566596,
 'lime_ris_max': 4844.12729015775,
 'std(lime_ris_max)': 440.3216594379406,
 't_exp_ris_mean': 12.154823964981588,
 'std(t_exp_ris_mean)': 37.47600220754622,
 'shap_ris_mean': 55.47708205546037,
 'std(shap_ris_mean)': 436.0290603370988,
 'lime_ris_mean': 13.907884768543969,
 'std(lime_ris_mean)': 112.63648374427316,
 't_exp_ros_max': 8006.055158737774,
 'std(t_exp_ros_max)': 816.2924956521439,
 'shap_ros_max': 142760.0849022648,
 'std(shap_ros_max)': 13108.270927413654,
 'lime_ros_max': 2545.4750695364783,
 'std(lime_ros_max)': 339.9660247839139,
 't_exp_ros_mean': 47.36210894756957,
 'std(t_exp_ros_mean)': 202.90278528200454,
 'shap_ros_mean': 301.42153886052915,
 'std(shap_ros_mean)': 2268.0636404183833,
 'lime_ros_mean': 18.434178537732173,
 'std(lime_ros_mean)': 95.00564060961614,
 'shap_kernel_ris_max': 610.0900769394282,
 's

In [42]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_diab
start= time.time()
print('RES --')
res_nn2_diab= stab.run_stability(nn2_model_diab, test_diab[0:125], labels_test_diab[0:125], 
                               descriptor_diab, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn2_diab

RES --
Progress: [████████████████████████████████████████] 125/125 - Est wait 00:0.01


--- 3013.91 seconds ---


{'texp_res': 2.02900661633277e-15,
 'shap_res': 1.193812447098184e-16,
 'shap_kernel_res': 1.2719202621569003e-16,
 'shap_exact_res': 1.193812447098184e-16,
 'lime_res': 6.520806856012869e-17,
 'itGd_res': 2.589462819655575e-15,
 'iXGd_res': 1.9679126e-06,
 'dLif_res': 1.9678562e-06,
 'lwrp_res': 1.986778e-06,
 'smoothG_res': 1.4578544366677986,
 'vanillaG_res': 5.8788432e-06,
 'GuidBprop_res': 5.8788432e-06,
 'occlusion_res': 1.1920929e-06}

In [44]:
# evaluate relative input/output stability -- using instances from train_diab
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_diab= stab.relative_stability(nn3_model_diab, test_diab[0:125], labels_test_diab[0:125], perturbation, 
                                        descriptor_diab, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn3_diab

RIS/ROS --
Progress: [████████████████████████████████████████] 125/125 - Est wait 00:0.07


--- 1956.86 seconds ---


{'t_exp_ris_max': 2131.6074304299964,
 'std(t_exp_ris_max)': 318.4889830770113,
 'shap_ris_max': 1624.7006385239806,
 'std(shap_ris_max)': 228.13915683390815,
 'lime_ris_max': 42.2062331568409,
 'std(lime_ris_max)': 6.850367659920354,
 't_exp_ris_mean': 32.44864868042923,
 'std(t_exp_ris_mean)': 91.47840683203378,
 'shap_ris_mean': 29.188876505984187,
 'std(shap_ris_mean)': 67.94521468628194,
 'lime_ris_mean': 2.1687669430057386,
 'std(lime_ris_mean)': 3.1107801387931193,
 't_exp_ros_max': 2510045.695035585,
 'std(t_exp_ros_max)': 223615.6231389854,
 'shap_ros_max': 117902.12268282736,
 'std(shap_ros_max)': 10975.647320792743,
 'lime_ros_max': 74302.3336401072,
 'std(lime_ros_max)': 6679.359195820732,
 't_exp_ros_mean': 3514.069564445541,
 'std(t_exp_ros_mean)': 34405.4123493751,
 'shap_ros_mean': 517.0297215785002,
 'std(shap_ros_mean)': 3466.3028797409956,
 'lime_ros_mean': 124.32673350476745,
 'std(lime_ros_mean)': 1066.3834785709835,
 'shap_kernel_ris_max': 1624.7006385228135,
 'st

In [46]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_diab
start= time.time()
print('RES --')
res_nn3_diab= stab.run_stability(nn3_model_diab, test_diab[0:125], labels_test_diab[0:125], 
                               descriptor_diab, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn3_diab

RES --
Progress: [████████████████████████████████████████] 125/125 - Est wait 00:0.00


--- 1747.06 seconds ---


{'texp_res': 5.024315033919233e-15,
 'shap_res': 1.2490611347050096e-16,
 'shap_kernel_res': 1.1775693440128312e-16,
 'shap_exact_res': 1.2490611347050096e-16,
 'lime_res': 4.4439065127221604e-17,
 'itGd_res': 5.402578481197087e-15,
 'iXGd_res': 4.2749457e-06,
 'dLif_res': 4.318047e-06,
 'lwrp_res': 4.268292e-06,
 'smoothG_res': 4.763623400767701,
 'vanillaG_res': 1.36295375e-05,
 'GuidBprop_res': 1.36295375e-05,
 'occlusion_res': 2.3856755e-06}

# 6. German Credit

In [61]:
ger_path= 'data/german/processed/'

train_ger= pd.read_csv(ger_path + 'X_train.csv')
test_ger = pd.read_csv(ger_path + 'X_test.csv')
labels_train_ger= pd.read_csv(ger_path + 'y_train.csv')
labels_test_ger = pd.read_csv(ger_path + 'y_test.csv')

In [63]:
numeric_columns_ger= ['Age', 'Job', 'Credit amount', 'Duration']

categor_columns_ger= list(filter(lambda x:x not in numeric_columns_ger, train_ger.columns))

categor_columns_ger

['Sex_male',
 'Housing_free',
 'Housing_own',
 'Housing_rent',
 'Saving accounts_little',
 'Saving accounts_moderate',
 'Saving accounts_quite rich',
 'Saving accounts_rich',
 'Checking account_little',
 'Checking account_moderate',
 'Checking account_rich',
 'Purpose_business',
 'Purpose_car',
 'Purpose_domestic appliances',
 'Purpose_education',
 'Purpose_furniture/equipment',
 'Purpose_radio/TV',
 'Purpose_repairs',
 'Purpose_vacation/others']

In [65]:
# 2024-05-29 17:41:34,069 Best: 0.668612 using {'batch_size': 128, 'lr': 0.01, 'max_epochs': 128, 
#                               'module__n_features': 23, 'module__n_neurons': 16, 'module__nonlin': Tanh()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_ger= MLPClassifier(batch_size= 128,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=1024,
                            hidden_layer_sizes=(16,),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_ger.fit(train_ger, labels_train_ger.values.ravel())

acc_nn1_ger= sklearn.metrics.accuracy_score(labels_test_ger.values.ravel(), nn1_model_ger.predict(test_ger))
acc_nn1_ger

0.63

In [67]:
# 2024-05-31 17:28:51,852 Best: 0.675973 using {'batch_size': 32, 'lr': 0.02, 'max_epochs': 16, 
#                               'module__n_features': 23, 'module__n_neurons': 256, 'module__nonlin': Tanh()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_ger= MLPClassifier(batch_size= 32,
                            learning_rate='constant', learning_rate_init= 0.02,
                            max_iter=64,
                            hidden_layer_sizes=(256, 256),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_ger.fit(train_ger, labels_train_ger.values.ravel())

acc_nn2_ger= sklearn.metrics.accuracy_score(labels_test_ger.values.ravel(), nn2_model_ger.predict(test_ger))
acc_nn2_ger

0.66

In [69]:
# 2024-06-03 16:27:03,452 Best: 0.668002 using {'batch_size': 64, 'lr': 0.02, 'max_epochs': 32, 
#                               'module__n_features': 23, 'module__n_neurons': 16, 'module__nonlin': Tanh()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_ger= MLPClassifier(#batch_size= 64,
                            learning_rate='constant', learning_rate_init= 0.02,
                            max_iter=512,
                            hidden_layer_sizes=(64, 64, 64),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='sgd')

nn3_model_ger.fit(train_ger, labels_train_ger.values.ravel())

acc_nn3_ger= sklearn.metrics.accuracy_score(labels_test_ger.values.ravel(), nn3_model_ger.predict(test_ger))
acc_nn3_ger

0.65

In [91]:
#

In [92]:
# 2024-06-03 16:27:03,452 Best: 0.668002 using {'batch_size': 64, 'lr': 0.02, 'max_epochs': 32, 
#                               'module__n_features': 23, 'module__n_neurons': 16, 'module__nonlin': Tanh()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_ger= MLPClassifier(#batch_size= 64,
                            learning_rate='constant', learning_rate_init= 0.02,
                            max_iter=512,
                            hidden_layer_sizes=(64, 64, 64),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_ger.fit(train_ger, labels_train_ger.values.ravel())

acc_nn3_ger= sklearn.metrics.accuracy_score(labels_test_ger.values.ravel(), nn3_model_ger.predict(test_ger))
acc_nn3_ger

0.58

In [75]:
# definitions ---- 154 samples from test dataset

# get n and m parameters from test and labels_test
n_ger, m_ger= texp.get_n_m_sizes(test_ger.loc[0:125], labels_test_ger[0:125])

# conversion of test_ox data back to tensor as required by benchmarking methods
# explanations will be only done over test dataset
tn_test= torch.from_numpy((test_ger.loc[0:125]).values)


# define a descriptor to synthetic data
descriptor_ger= dict()

# h_min_dist_ox comes from the entire train data for capturing a model's space characteristic
h_min_dist_ger= texp.get_minimum_distance(train_ger)

# T-Exp explanation settings
descriptor_ger['h_min']= h_min_dist_ger
descriptor_ger['h_max']= 1
descriptor_ger['jacobian_eps']= 1e-3
descriptor_ger['max_itr']= 30
descriptor_ger['ohe_delta']= 0.2

# perturbation settings used for the stability metric
descriptor_ger['num_samples']= 30
descriptor_ger['num_perts']= 10   # RIS/ROS
descriptor_ger['pert_max_distance']= (h_min_dist_ger/2)
descriptor_ger['num_runs']= 10    # RES
descriptor_ger['feature_metadata']= ['c'] * n_ger
descriptor_ger['p_norm']= 2
descriptor_ger['eps_norm']= 1e-6
descriptor_ger['top_k']= 0
descriptor_ger['mask']= generate_mask(tn_test[0].reshape(-1), descriptor_ger['top_k'])

In [77]:
if (h_min_dist_ger< 0.01): h_min_dist_ger= 0.01

h_min_dist_ger

0.01

In [79]:
# ---- 154 samples from test dataset

# evaluate relative input/output stability -- using instances from train_ger
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_ger= stab.relative_stability(nn1_model_ger, test_ger[0:125], labels_test_ger[0:125], perturbation, 
                                         descriptor_ger, cat_fts=categor_columns_ger, 
                                         train_data=train_ger, labels_train=labels_train_ger,
                                         is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn1_ger

RIS/ROS --
Progress: [████████████████████████████████████████] 125/125 - Est wait 00:0.00


--- 6745.39 seconds ---


{'t_exp_ris_max': 10928.96402272606,
 'std(t_exp_ris_max)': 1191.4580009520598,
 'shap_ris_max': 157776.74503838195,
 'std(shap_ris_max)': 39698.477008149974,
 'lime_ris_max': 20.9782146681933,
 'std(lime_ris_max)': 2.5980526265573096,
 't_exp_ris_mean': 141.0966203410958,
 'std(t_exp_ris_mean)': 434.97816172502206,
 'shap_ris_mean': 5271.538257889595,
 'std(shap_ris_mean)': 13104.380948144328,
 'lime_ris_mean': 0.7623534041400567,
 'std(lime_ris_mean)': 1.0668736030411714,
 't_exp_ros_max': 35529.52965740194,
 'std(t_exp_ros_max)': 4962.970182100838,
 'shap_ros_max': 16302867.941217113,
 'std(shap_ros_max)': 1514206.527475598,
 'lime_ros_max': 342.57724294342654,
 'std(lime_ros_max)': 38.77377945429488,
 't_exp_ros_mean': 440.91106091281625,
 'std(t_exp_ros_mean)': 1206.5534381437074,
 'shap_ros_mean': 29103.971465345207,
 'std(shap_ros_mean)': 167100.63501900018,
 'lime_ros_mean': 2.9420872738073136,
 'std(lime_ros_mean)': 5.211594858878449,
 'shap_kernel_ris_max': 721.5471985977521,

In [81]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_ger
start= time.time()
print('RES --')
res_nn1_ger= stab.run_stability(nn1_model_ger, test_ger[0:125], labels_test_ger[0:125], 
                                descriptor_ger, cat_fts=categor_columns_ger, 
                                train_data=train_ger, labels_train=labels_train_ger,
                                is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn1_ger

RES --
Progress: [████████████████████████████████████████] 125/125 - Est wait 00:0.00


--- 6118.27 seconds ---


{'texp_res': 2.158397618328465e-15,
 'shap_res': 0.19155547499353812,
 'shap_kernel_res': 0.07790938918481556,
 'shap_exact_res': '--',
 'lime_res': 4.501097670045862e-17,
 'itGd_res': 2.094764613337708e-15,
 'iXGd_res': 1.2949444e-06,
 'dLif_res': 1.2949444e-06,
 'lwrp_res': 1.2728069e-06,
 'smoothG_res': 1.5841842207106689,
 'vanillaG_res': 3.1137456e-06,
 'GuidBprop_res': 3.1137456e-06,
 'occlusion_res': 1.2502776e-06}

In [83]:
# evaluate relative input/output stability -- using instances from train_ger
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_ger= stab.relative_stability(nn2_model_ger, test_ger[0:125], labels_test_ger[0:125], perturbation, 
                                         descriptor_ger, cat_fts=categor_columns_ger, 
                                         train_data=train_ger, labels_train=labels_train_ger,
                                         is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn2_ger

RIS/ROS --
Progress: [████████████████████████████████████████] 125/125 - Est wait 00:0.09


--- 16887.15 seconds ---


{'t_exp_ris_max': 5078.031816313588,
 'std(t_exp_ris_max)': 578.6231751298299,
 'shap_ris_max': 115455.10340950338,
 'std(shap_ris_max)': 16210.53726701642,
 'lime_ris_max': 154.43400230648425,
 'std(lime_ris_max)': 15.830392712691694,
 't_exp_ris_mean': 93.2836211839482,
 'std(t_exp_ris_mean)': 172.9028188736024,
 'shap_ris_mean': 1629.0465934078911,
 'std(shap_ris_mean)': 5580.698621878859,
 'lime_ris_mean': 1.4365625143569059,
 'std(lime_ris_mean)': 5.519741089905321,
 't_exp_ros_max': 4731699.419095319,
 'std(t_exp_ros_max)': 421400.8067138193,
 'shap_ros_max': 2586433.8530189577,
 'std(shap_ros_max)': 422285.74350095115,
 'lime_ros_max': 4628.033128575631,
 'std(lime_ros_max)': 427.6614008966281,
 't_exp_ros_mean': 5896.769209619859,
 'std(t_exp_ros_mean)': 45242.39964233195,
 'shap_ros_mean': 22765.352486759188,
 'std(shap_ros_mean)': 68201.78748130331,
 'lime_ros_mean': 15.962186822958724,
 'std(lime_ros_mean)': 51.374778099886335,
 'shap_kernel_ris_max': 338.0893962642408,
 'st

In [85]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_ger
start= time.time()
print('RES --')
res_nn2_ger= stab.run_stability(nn2_model_ger, test_ger[0:125], labels_test_ger[0:125], 
                                descriptor_ger, cat_fts=categor_columns_ger, 
                                train_data=train_ger, labels_train=labels_train_ger,
                                is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn2_ger

RES --
Progress: [████████████████████████████████████████] 125/125 - Est wait 00:0.07


--- 15099.41 seconds ---


{'texp_res': 6.066505582541061e-16,
 'shap_res': 0.1296082751987696,
 'shap_kernel_res': 0.04422173062972382,
 'shap_exact_res': '--',
 'lime_res': 2.1404688228901976e-17,
 'itGd_res': 3.978256139440565e-15,
 'iXGd_res': 2.095561e-06,
 'dLif_res': 2.081954e-06,
 'lwrp_res': 2.2429094e-06,
 'smoothG_res': 1.2355345412428789,
 'vanillaG_res': 4.381651e-06,
 'GuidBprop_res': 4.381651e-06,
 'occlusion_res': 2.1851424e-06}

In [87]:
# evaluate relative input/output stability -- using instances from train_ger
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_ger= stab.relative_stability(nn3_model_ger, test_ger[0:125], labels_test_ger[0:125], perturbation, 
                                         descriptor_ger, cat_fts=categor_columns_ger, 
                                         train_data=train_ger, labels_train=labels_train_ger,
                                         is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn3_ger

RIS/ROS --
Progress: [████████████████████████████████████████] 125/125 - Est wait 00:0.04


--- 7955.23 seconds ---


{'t_exp_ris_max': 3790.958814249851,
 'std(t_exp_ris_max)': 531.2054523145027,
 'shap_ris_max': 170200.7352796431,
 'std(shap_ris_max)': 24309.047682754586,
 'lime_ris_max': 67.38490149507642,
 'std(lime_ris_max)': 6.9934142604863805,
 't_exp_ris_mean': 73.97453301053173,
 'std(t_exp_ris_mean)': 155.1164966934014,
 'shap_ris_mean': 2460.801503210533,
 'std(shap_ris_mean)': 7207.220830804817,
 'lime_ris_mean': 1.3740591388585601,
 'std(lime_ris_mean)': 2.500626287283715,
 't_exp_ros_max': 113800.51902471331,
 'std(t_exp_ros_max)': 10568.398085867997,
 'shap_ros_max': 982791.0803649864,
 'std(shap_ros_max)': 146797.60144268285,
 'lime_ros_max': 3076.4847484618663,
 'std(lime_ros_max)': 284.28097537441334,
 't_exp_ros_mean': 338.6217792008884,
 'std(t_exp_ros_mean)': 1153.830844601493,
 'shap_ros_mean': 7341.250022877981,
 'std(shap_ros_mean)': 22362.233396289685,
 'lime_ros_mean': 7.375341564127951,
 'std(lime_ros_mean)': 29.074350088573254,
 'shap_kernel_ris_max': 156.20006708997698,
 '

In [89]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_ger
start= time.time()
print('RES --')
res_nn3_ger= stab.run_stability(nn3_model_ger, test_ger[0:125], labels_test_ger[0:125], 
                                descriptor_ger, cat_fts=categor_columns_ger, 
                                train_data=train_ger, labels_train=labels_train_ger,
                                is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn3_ger

RES --
Progress: [████████████████████████████████████████] 125/125 - Est wait 00:0.07


--- 7473.79 seconds ---


{'texp_res': 1.0934002113791248e-15,
 'shap_res': 0.1737977847960094,
 'shap_kernel_res': 0.06317131563667644,
 'shap_exact_res': '--',
 'lime_res': 3.388335612894791e-17,
 'itGd_res': 2.042629342666091e-15,
 'iXGd_res': 1.0990806e-06,
 'dLif_res': 1.397854e-06,
 'lwrp_res': 1.1071087e-06,
 'smoothG_res': 1.567000103730011,
 'vanillaG_res': 2.506586e-06,
 'GuidBprop_res': 2.506586e-06,
 'occlusion_res': 1.0409503e-06}

# 7. HELOC

In [6]:
hel_path= 'data/heloc/processed/'

train_hel= pd.read_csv(hel_path + 'X_train.csv')
test_hel = pd.read_csv(hel_path + 'X_test.csv')
labels_train_hel= pd.read_csv(hel_path + 'y_train.csv')
labels_test_hel = pd.read_csv(hel_path + 'y_test.csv')

In [50]:
categor_columns_hel= [
    'MaxDelq2PublicRecLast12M_0.0','MaxDelq2PublicRecLast12M_1.0','MaxDelq2PublicRecLast12M_2.0',
    'MaxDelq2PublicRecLast12M_3.0','MaxDelq2PublicRecLast12M_4.0','MaxDelq2PublicRecLast12M_5.0',
    'MaxDelq2PublicRecLast12M_6.0','MaxDelq2PublicRecLast12M_7.0','MaxDelq2PublicRecLast12M_9.0',
    'MaxDelqEver_2.0','MaxDelqEver_3.0','MaxDelqEver_4.0','MaxDelqEver_5.0','MaxDelqEver_6.0',
    'MaxDelqEver_7.0','MaxDelqEver_8.0']

categor_columns_hel= np.asarray(categor_columns_hel)

In [8]:
# 2024-05-30 09:37:27,988 Best: 0.802750 using {'batch_size': 16, 'lr': 0.001, 'max_epochs': 128, 
#                               'module__n_features': 37, 'module__n_neurons': 256, 'module__nonlin': Tanh()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_hel= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.001,
                            max_iter=128,
                            hidden_layer_sizes=(256,),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_hel.fit(train_hel, labels_train_hel.values.ravel())

acc_nn1_hel= sklearn.metrics.accuracy_score(labels_test_hel.values.ravel(), nn1_model_hel.predict(test_hel))
acc_nn1_hel

0.7169620253164557

In [9]:
# 2024-06-01 04:05:45,253 Best: 0.802888 using {'batch_size': 16, 'lr': 0.001, 'max_epochs': 128, 
#                               'module__n_features': 37, 'module__n_neurons': 16, 'module__nonlin': Tanh()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_hel= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.001,
                            max_iter=512,
                            hidden_layer_sizes=(16, 16),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_hel.fit(train_hel, labels_train_hel.values.ravel())

acc_nn2_hel= sklearn.metrics.accuracy_score(labels_test_hel.values.ravel(), nn2_model_hel.predict(test_hel))
acc_nn2_hel

0.7007594936708861

In [10]:
# 2024-06-04 04:15:54,809 Best: 0.802922 using {'batch_size': 64, 'lr': 0.001, 'max_epochs': 16, 
#                               'module__n_features': 37, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_hel= MLPClassifier(#batch_size= 64,
                            learning_rate='constant', learning_rate_init= 0.001,
                            max_iter=512,
                            hidden_layer_sizes=(256, 256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_hel.fit(train_hel, labels_train_hel.values.ravel())

acc_nn3_hel= sklearn.metrics.accuracy_score(labels_test_hel.values.ravel(), nn3_model_hel.predict(test_hel))
acc_nn3_hel

0.660253164556962

In [11]:
# definitions ---- 499 samples from test dataset

# get n and m parameters from test and labels_test
n_hel, m_hel= texp.get_n_m_sizes(test_hel.loc[0:498], labels_test_hel[0:498])

# conversion of test_ox data back to tensor as required by benchmarking methods
# explanations will be only done over test dataset
tn_test= torch.from_numpy((test_hel.loc[0:498]).values)


# define a descriptor to synthetic data
descriptor_hel= dict()

# h_min_dist_ox comes from the entire train data for capturing a model's space characteristic
h_min_dist_hel= texp.get_minimum_distance(train_hel)

# T-Exp explanation settings
descriptor_hel['h_min']= h_min_dist_hel
descriptor_hel['h_max']= 1
descriptor_hel['jacobian_eps']= 1e-3
descriptor_hel['max_itr']= 30
descriptor_hel['ohe_delta']= 0.2

# perturbation settings used for the stability metric
descriptor_hel['num_samples']= 30
descriptor_hel['num_perts']= 10   # RIS/ROS
descriptor_hel['pert_max_distance']= (h_min_dist_hel/2)
descriptor_hel['num_runs']= 10    # RES
descriptor_hel['feature_metadata']= ['c'] * n_hel
descriptor_hel['p_norm']= 2
descriptor_hel['eps_norm']= 1e-6
descriptor_hel['top_k']= 0
descriptor_hel['mask']= generate_mask(tn_test[0].reshape(-1), descriptor_hel['top_k'])

In [13]:
if (h_min_dist_hel< 0.01): h_min_dist_hel= 0.01
    
h_min_dist_hel

0.026965918874184168

In [51]:
#

In [52]:
# ---- 499 samples from test dataset

# evaluate relative input/output stability -- using instances from train_hel
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_hel= stab.relative_stability(nn1_model_hel, test_hel[0:498], labels_test_hel[0:498], perturbation, 
                                        descriptor_hel, cat_fts=categor_columns_hel, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn1_hel

RIS/ROS --


AttributeError: 'list' object has no attribute 'shape'

In [38]:
#

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_hel
start= time.time()
print('RES --')
res_nn1_hel= stab.run_stability(nn1_model_hel, test_hel[0:498], labels_test_hel[0:498], 
                               descriptor_hel, cat_fts=categor_columns_hel, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn1_hel

In [ ]:
#

In [ ]:
# evaluate relative input/output stability -- using instances from train_hel
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_hel= stab.relative_stability(nn2_model_hel, test_hel[0:498], labels_test_hel[0:498], perturbation, 
                                        descriptor_hel, cat_fts=categor_columns_hel, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn2_hel

In [ ]:
#

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_hel
start= time.time()
print('RES --')
res_nn2_hel= stab.run_stability(nn2_model_hel, test_hel[0:498], labels_test_hel[0:498], 
                               descriptor_hel, cat_fts=categor_columns_hel, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn2_hel

In [ ]:
#

In [ ]:
# evaluate relative input/output stability -- using instances from train_hel
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_hel= stab.relative_stability(nn3_model_hel, test_hel[0:498], labels_test_hel[0:498], perturbation, 
                                        descriptor_hel, cat_fts=categor_columns_hel, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn3_hel

In [ ]:
#

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_hel
start= time.time()
print('RES --')
res_nn3_hel= stab.run_stability(nn3_model_hel, test_hel[0:498], labels_test_hel[0:498], 
                               descriptor_hel, cat_fts=categor_columns_hel, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn3_hel

In [ ]:
#

# 8. HIGGS

In [ ]:
hig_path= 'data/higgs/processed/'

train_hig= pd.read_csv(hig_path + 'X_train.csv')
test_hig = pd.read_csv(hig_path + 'X_test.csv')
labels_train_hig= pd.read_csv(hig_path + 'y_train.csv')
labels_test_hig = pd.read_csv(hig_path + 'y_test.csv')

In [ ]:
# 2024-05-29 23:24:10,338 Best: 0.908399 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_hig= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=256,
                            hidden_layer_sizes=(256,),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_hig.fit(train_hig, labels_train_hig.values.ravel())

acc_nn1_hig= sklearn.metrics.accuracy_score(labels_test_hig.values.ravel(), nn1_model_hig.predict(test_hig))
acc_nn1_hig

In [ ]:
# 2024-05-31 17:29:14,888 Best: 0.912651 using {'batch_size': 64, 'lr': 0.01, 'max_epochs': 64, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_hig= MLPClassifier(batch_size= 64,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_hig.fit(train_hig, labels_train_hig.values.ravel())

acc_nn2_hig= sklearn.metrics.accuracy_score(labels_test_hig.values.ravel(), nn2_model_hig.predict(test_hig))
acc_nn2_hig

In [ ]:
# 2024-06-03 16:27:56,409 Best: 0.905554 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_hig= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_hig.fit(train_hig, labels_train_hig.values.ravel())

acc_nn3_hig= sklearn.metrics.accuracy_score(labels_test_hig.values.ravel(), nn3_model_hig.predict(test_hig))
acc_nn3_hig

In [ ]:
# definitions ---- 644 samples from test dataset

# get n and m parameters from test and labels_test
n_hig, m_hig= texp.get_n_m_sizes(test_hig.loc[0:643], labels_test_hig[0:643])

# conversion of test_ox data back to tensor as required by benchmarking methods
# explanations will be only done over test dataset
tn_test= torch.from_numpy((test_hig.loc[0:643]).values)


# define a descriptor to synthetic data
descriptor_hig= dict()

# h_min_dist_ox comes from the entire train data for capturing a model's space characteristic
h_min_dist_hig= texp.get_minimum_distance(train_hig)

# T-Exp explanation settings
descriptor_hig['h_min']= h_min_dist_hig
descriptor_hig['h_max']= 1
descriptor_hig['jacobian_eps']= 1e-3
descriptor_hig['max_itr']= 30
descriptor_hig['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_hig['num_samples']= 30
descriptor_hig['num_perts']= 10   # RIS/ROS
descriptor_hig['pert_max_distance']= (h_min_dist_hig/2)
descriptor_hig['num_runs']= 10    # RES
descriptor_hig['feature_metadata']= ['c'] * n_hig
descriptor_hig['p_norm']= 2
descriptor_hig['eps_norm']= 1e-6
descriptor_hig['top_k']= 0
descriptor_hig['mask']= generate_mask(tn_test[0].reshape(-1), descriptor_hig['top_k'])

In [ ]:
# ---- 644 samples from test dataset

# evaluate relative input/output stability -- using instances from train_hig
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_hig= stab.relative_stability(nn1_model_hig, test_hig[0:643], labels_test_hig[0:643], perturbation, 
                                        descriptor_hig, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn1_hig

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_hig
start= time.time()
print('RES --')
res_nn1_hig= stab.run_stability(nn1_model_hig, test_hig[0:643], labels_test_hig[0:643], 
                               descriptor_hig, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn1_hig

In [ ]:
# evaluate relative input/output stability -- using instances from train_hig
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_hig= stab.relative_stability(nn2_model_hig, test_hig[0:643], labels_test_hig[0:643], perturbation, 
                                        descriptor_hig, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn2_hig

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_hig
start= time.time()
print('RES --')
res_nn2_hig= stab.run_stability(nn2_model_hig, test_hig[0:643], labels_test_hig[0:643], 
                               descriptor_hig, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn2_hig

In [ ]:
# evaluate relative input/output stability -- using instances from train_hig
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_hig= stab.relative_stability(nn3_model_hig, test_hig[0:643], labels_test_hig[0:643], perturbation, 
                                        descriptor_hig, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn3_hig

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_hig
start= time.time()
print('RES --')
res_nn3_hig= stab.run_stability(nn3_model_hig, test_hig[0:643], labels_test_hig[0:643], 
                               descriptor_hig, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn3_hig

# 9. Independent

In [18]:
indep_path= 'data/independent/processed/'

train_indep= pd.read_csv(indep_path + 'X_train.csv')
test_indep = pd.read_csv(indep_path + 'X_test.csv')
labels_train_indep= pd.read_csv(indep_path + 'y_train.csv')
labels_test_indep = pd.read_csv(indep_path + 'y_test.csv')

In [19]:
# 2024-05-29 17:02:04,589 Best: 0.997288 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 32, 
#                               'module__n_features': 6, 'module__n_neurons': 256, 'module__nonlin': Tanh()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_indep= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=64,
                            hidden_layer_sizes=(256,),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_indep.fit(train_indep, labels_train_indep.values.ravel())

acc_nn1_indep= sklearn.metrics.accuracy_score(labels_test_indep.values.ravel(), nn1_model_indep.predict(test_indep))
acc_nn1_indep

1.0

In [20]:
# 2024-05-31 16:17:24,671 Best: 0.997288 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 16, 
#                               'module__n_features': 6, 'module__n_neurons': 32, 'module__nonlin': Tanh()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_indep= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=64,
                            hidden_layer_sizes=(32, 32),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_indep.fit(train_indep, labels_train_indep.values.ravel())

acc_nn2_indep= sklearn.metrics.accuracy_score(labels_test_indep.values.ravel(), nn2_model_indep.predict(test_indep))
acc_nn2_indep

0.9666666666666667

In [21]:
# 2024-06-03 15:02:42,848 Best: 0.997667 using {'batch_size': 64, 'lr': 0.01, 'max_epochs': 16, 
#                               'module__n_features': 6, 'module__n_neurons': 32, 'module__nonlin': Tanh()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_indep= MLPClassifier(batch_size= 64,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=64,
                            hidden_layer_sizes=(32, 32, 32),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_indep.fit(train_indep, labels_train_indep.values.ravel())

acc_nn3_indep= sklearn.metrics.accuracy_score(labels_test_indep.values.ravel(), nn3_model_indep.predict(test_indep))
acc_nn3_indep

1.0

In [22]:
# definitions ---- 60 samples from test dataset

# get n and m parameters from test and labels_test
n_indep, m_indep= texp.get_n_m_sizes(test_indep, labels_test_indep)

# conversion of test_ox data back to tensor as required by benchmarking methods
# explanations will be only done over test dataset
tn_test= torch.from_numpy(test_indep.values)


# define a descriptor to synthetic data
descriptor_indep= dict()

# h_min_dist_ox comes from the entire train data for capturing a model's space characteristic
h_min_dist_indep= texp.get_minimum_distance(train_indep)

# T-Exp explanation settings
descriptor_indep['h_min']= h_min_dist_indep
descriptor_indep['h_max']= 1
descriptor_indep['jacobian_eps']= 1e-3
descriptor_indep['max_itr']= 30
descriptor_indep['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_indep['num_samples']= 30
descriptor_indep['num_perts']= 10   # RIS/ROS
descriptor_indep['pert_max_distance']= (h_min_dist_indep/2)
descriptor_indep['num_runs']= 10    # RES
descriptor_indep['feature_metadata']= ['c'] * n_indep
descriptor_indep['p_norm']= 2
descriptor_indep['eps_norm']= 1e-6
descriptor_indep['top_k']= 0
descriptor_indep['mask']= generate_mask(tn_test[0].reshape(-1), descriptor_indep['top_k'])

In [23]:
h_min_dist_indep

0.1144753630789289

In [24]:
# ---- 60 samples from test dataset

# evaluate relative input/output stability -- using instances from train_indep
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_indep= stab.relative_stability(nn1_model_indep, test_indep, labels_test_indep, perturbation, 
                                        descriptor_indep, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn1_indep

RIS/ROS --
Progress: [████████████████████████████████████████] 60/60 - Est wait 00:0.06


--- 754.59 seconds ---


{'t_exp_ris_max': 114.38746341843554,
 'std(t_exp_ris_max)': 15.676762374387875,
 'shap_ris_max': 42631.47214824999,
 'std(shap_ris_max)': 5453.069484446505,
 'lime_ris_max': 17.12143926179611,
 'std(lime_ris_max)': 2.7747848645745776,
 't_exp_ris_mean': 2.725385323169389,
 'std(t_exp_ris_mean)': 4.471007058399122,
 'shap_ris_mean': 197.16405134048725,
 'std(shap_ris_mean)': 1400.1214680694343,
 'lime_ris_mean': 1.0608378452984562,
 'std(lime_ris_mean)': 1.2771862593583372,
 't_exp_ros_max': 249672.27087442746,
 'std(t_exp_ros_max)': 32095.905291745043,
 'shap_ros_max': 392005.8549337078,
 'std(shap_ros_max)': 64342.85547624924,
 'lime_ros_max': 253562.64710422602,
 'std(lime_ros_max)': 32454.572880666714,
 't_exp_ros_mean': 1467.5934786273478,
 'std(t_exp_ros_mean)': 8395.975420973673,
 'shap_ros_mean': 1891.6921044473047,
 'std(shap_ros_mean)': 8648.389080548144,
 'lime_ros_mean': 544.0314348136429,
 'std(lime_ros_mean)': 3726.102458007639,
 'shap_kernel_ris_max': 1103.8485752460888,

In [26]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_indep
start= time.time()
print('RES --')
res_nn1_indep= stab.run_stability(nn1_model_indep, test_indep, labels_test_indep, 
                               descriptor_indep, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn1_indep

RES --
Progress: [████████████████████████████████████████] 60/60 - Est wait 00:0.01


--- 608.7 seconds ---


{'texp_res': 5.6217203786222585e-15,
 'shap_res': 1.3019674641152052e-16,
 'shap_kernel_res': 1.2719202621569003e-16,
 'shap_exact_res': 1.3019674641152052e-16,
 'lime_res': 8.328592404739402e-17,
 'itGd_res': 2.6651134375294657e-15,
 'iXGd_res': 1.652098e-06,
 'dLif_res': 1.652098e-06,
 'lwrp_res': 1.508186e-06,
 'smoothG_res': 0.7390290859876985,
 'vanillaG_res': 2.8622644e-06,
 'GuidBprop_res': 2.8622644e-06,
 'occlusion_res': 1.7192608e-06}

In [28]:
# evaluate relative input/output stability -- using instances from train_indep
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_indep= stab.relative_stability(nn2_model_indep, test_indep, labels_test_indep, perturbation, 
                                        descriptor_indep, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn2_indep

RIS/ROS --
Progress: [████████████████████████████████████████] 60/60 - Est wait 00:0.00


--- 658.72 seconds ---


{'t_exp_ris_max': 47.392604190517005,
 'std(t_exp_ris_max)': 6.891011112512014,
 'shap_ris_max': 15842.365460039046,
 'std(shap_ris_max)': 3454.7674799497304,
 'lime_ris_max': 432.41522047119594,
 'std(lime_ris_max)': 55.559156032240416,
 't_exp_ris_mean': 2.760798114604981,
 'std(t_exp_ris_mean)': 2.9866480707372665,
 'shap_ris_mean': 354.87038182361056,
 'std(shap_ris_mean)': 1152.4514230451582,
 'lime_ris_mean': 5.824885518617606,
 'std(lime_ris_mean)': 21.81277087521687,
 't_exp_ros_max': 861.0666074120741,
 'std(t_exp_ros_max)': 172.5279093523794,
 'shap_ros_max': 348801.26302727027,
 'std(shap_ros_max)': 45309.726541525895,
 'lime_ros_max': 9162.377586056957,
 'std(lime_ros_max)': 1233.7887454609786,
 't_exp_ros_mean': 17.547321133870064,
 'std(t_exp_ros_mean)': 48.4839975829335,
 'shap_ros_mean': 2398.649870935022,
 'std(shap_ros_mean)': 15886.041878332335,
 'lime_ros_mean': 57.039020068726714,
 'std(lime_ros_mean)': 248.72570616660008,
 'shap_kernel_ris_max': 226.3510955351302,

In [30]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_indep
start= time.time()
print('RES --')
res_nn2_indep= stab.run_stability(nn2_model_indep, test_indep, labels_test_indep, 
                               descriptor_indep, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn2_indep

RES --
Progress: [████████████████████████████████████████] 60/60 - Est wait 00:0.09


--- 565.76 seconds ---


{'texp_res': 5.17894423319163e-15,
 'shap_res': 1.1102230246251565e-16,
 'shap_kernel_res': 1.1775693440128312e-16,
 'shap_exact_res': 1.1102230246251565e-16,
 'lime_res': 8.326785621649077e-17,
 'itGd_res': 2.083194168503685e-15,
 'iXGd_res': 1.3486991e-06,
 'dLif_res': 1.3486991e-06,
 'lwrp_res': 1.5078915e-06,
 'smoothG_res': 0.7615362707683194,
 'vanillaG_res': 2.3511748e-06,
 'GuidBprop_res': 2.3511748e-06,
 'occlusion_res': 1.4354699e-06}

In [32]:
# evaluate relative input/output stability -- using instances from train_indep
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_indep= stab.relative_stability(nn3_model_indep, test_indep, labels_test_indep, perturbation, 
                                        descriptor_indep, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn3_indep

RIS/ROS --
Progress: [████████████████████████████████████████] 60/60 - Est wait 00:0.06


--- 698.54 seconds ---


{'t_exp_ris_max': 471.3125160698543,
 'std(t_exp_ris_max)': 59.99075387245171,
 'shap_ris_max': 5308.869288431973,
 'std(shap_ris_max)': 1049.4505518163523,
 'lime_ris_max': 174.85970184321707,
 'std(lime_ris_max)': 26.509375003595025,
 't_exp_ris_mean': 9.015238479669561,
 'std(t_exp_ris_mean)': 21.968339466587256,
 'shap_ris_mean': 114.61003980172566,
 'std(shap_ris_mean)': 352.77359357888304,
 'lime_ris_mean': 5.761384279081517,
 'std(lime_ris_mean)': 11.743967519897593,
 't_exp_ros_max': 9307.432437811487,
 'std(t_exp_ros_max)': 1191.7296632523837,
 'shap_ros_max': 55800.84894830322,
 'std(shap_ros_max)': 10612.553370595515,
 'lime_ros_max': 157.62624729124363,
 'std(lime_ros_max)': 33.715010159349944,
 't_exp_ros_mean': 35.81393768483659,
 'std(t_exp_ros_mean)': 206.8943155384605,
 'shap_ros_mean': 724.9077686955685,
 'std(shap_ros_mean)': 2731.64156290393,
 'lime_ros_mean': 5.532615523577732,
 'std(lime_ros_mean)': 9.57218930199032,
 'shap_kernel_ris_max': 1042.9105279855773,
 's

In [34]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_indep
start= time.time()
print('RES --')
res_nn3_indep= stab.run_stability(nn3_model_indep, test_indep, labels_test_indep, 
                               descriptor_indep, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn3_indep

RES --
Progress: [████████████████████████████████████████] 60/60 - Est wait 00:0.06


--- 633.88 seconds ---


{'texp_res': 6.158736452937403e-15,
 'shap_res': 1.2719202621569003e-16,
 'shap_kernel_res': 1.1775693440128312e-16,
 'shap_exact_res': 1.2719202621569003e-16,
 'lime_res': 8.777083671441753e-17,
 'itGd_res': 1.8477795749834654e-15,
 'iXGd_res': 1.1740754e-06,
 'dLif_res': 1.1935821e-06,
 'lwrp_res': 1.3513308e-06,
 'smoothG_res': 0.6179273962124862,
 'vanillaG_res': 2.9103078e-06,
 'GuidBprop_res': 2.9103078e-06,
 'occlusion_res': 1.2686131e-06}

# 10. LSA - Law School Admission

In [ ]:
lsa_path= 'data/law_school_admission/processed/'

train_lsa= pd.read_csv(lsa_path + 'X_train.csv')
test_lsa = pd.read_csv(lsa_path + 'X_test.csv')
labels_train_lsa= pd.read_csv(lsa_path + 'y_train.csv')
labels_test_lsa = pd.read_csv(lsa_path + 'y_test.csv')

In [ ]:
# 2024-05-29 23:24:10,338 Best: 0.908399 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_lsa= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=256,
                            hidden_layer_sizes=(256,),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_lsa.fit(train_lsa, labels_train_lsa.values.ravel())

acc_nn1_lsa= sklearn.metrics.accuracy_score(labels_test_lsa.values.ravel(), nn1_model_lsa.predict(test_lsa))
acc_nn1_lsa

In [ ]:
# 2024-05-31 17:29:14,888 Best: 0.912651 using {'batch_size': 64, 'lr': 0.01, 'max_epochs': 64, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_lsa= MLPClassifier(batch_size= 64,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_lsa.fit(train_lsa, labels_train_lsa.values.ravel())

acc_nn2_lsa= sklearn.metrics.accuracy_score(labels_test_lsa.values.ravel(), nn2_model_lsa.predict(test_lsa))
acc_nn2_lsa

In [ ]:
# 2024-06-03 16:27:56,409 Best: 0.905554 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_lsa= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_lsa.fit(train_lsa, labels_train_lsa.values.ravel())

acc_nn3_lsa= sklearn.metrics.accuracy_score(labels_test_lsa.values.ravel(), nn3_model_lsa.predict(test_lsa))
acc_nn3_lsa

In [ ]:
# definitions ---- 574 samples from test dataset

# get n and m parameters from test and labels_test
n_lsa, m_lsa= texp.get_n_m_sizes(test_lsa.loc[0:573], labels_test_lsa[0:573])

# conversion of test_ox data back to tensor as required by benchmarking methods
# explanations will be only done over test dataset
tn_test= torch.from_numpy((test_lsa.loc[0:573]).values)


# define a descriptor to synthetic data
descriptor_lsa= dict()

# h_min_dist_ox comes from the entire train data for capturing a model's space characteristic
h_min_dist_lsa= texp.get_minimum_distance(train_lsa)

# T-Exp explanation settings
descriptor_lsa['h_min']= h_min_dist_lsa
descriptor_lsa['h_max']= 1
descriptor_lsa['jacobian_eps']= 1e-3
descriptor_lsa['max_itr']= 30
descriptor_lsa['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_lsa['num_samples']= 30
descriptor_lsa['num_perts']= 10   # RIS/ROS
descriptor_lsa['pert_max_distance']= (h_min_dist_lsa/2)
descriptor_lsa['num_runs']= 10    # RES
descriptor_lsa['feature_metadata']= ['c'] * n_lsa
descriptor_lsa['p_norm']= 2
descriptor_lsa['eps_norm']= 1e-6
descriptor_lsa['top_k']= 0
descriptor_lsa['mask']= generate_mask(tn_test[0].reshape(-1), descriptor_lsa['top_k'])

In [ ]:
# ---- 574 samples from test dataset

# evaluate relative input/output stability -- using instances from train_lsa
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_lsa= stab.relative_stability(nn1_model_lsa, test_lsa[0:573], labels_test_lsa[0:573], perturbation, 
                                        descriptor_lsa, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn1_lsa

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_lsa
start= time.time()
print('RES --')
res_nn1_lsa= stab.run_stability(nn1_model_lsa, test_lsa[0:573], labels_test_lsa[0:573], 
                               descriptor_lsa, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn1_lsa

In [ ]:
# evaluate relative input/output stability -- using instances from train_lsa
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_lsa= stab.relative_stability(nn2_model_lsa, test_lsa[0:573], labels_test_lsa[0:573], perturbation, 
                                        descriptor_lsa, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn2_lsa

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_lsa
start= time.time()
print('RES --')
res_nn2_lsa= stab.run_stability(nn2_model_lsa, test_lsa[0:573], labels_test_lsa[0:573], 
                               descriptor_lsa, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn2_lsa

In [ ]:
# evaluate relative input/output stability -- using instances from train_lsa
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_lsa= stab.relative_stability(nn3_model_lsa, test_lsa[0:573], labels_test_lsa[0:573], perturbation, 
                                        descriptor_lsa, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn3_lsa

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using instances from train_lsa
start= time.time()
print('RES --')
res_nn3_lsa= stab.run_stability(nn3_model_lsa, test_lsa[0:573], labels_test_lsa[0:573], 
                               descriptor_lsa, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn3_lsa